# 04. Qdrant 적재 + 검색 함수 (데이터 v1)

> 담당: 성경 (임베딩 · Qdrant) · 기준: 청크 **A_fixed v1** (3,400개) · 모델 **KURE-v1** (1024차원, cosine)  
> 근거: `임베딩_AB테스트_결과보고서` (A/B 테스트 최종 실행 `20261110_062652_1d153850`)

**이 노트북이 하는 일**

| 순서 | 내용 |
|---|---|
| 1~7 | A/B 테스트에서 채점한 벡터(캐시)를 Docker Qdrant에 적재하고 확인 |
| 8~12 | 질문 임베딩(KURE-v1) · 지역 필터 · `search_policies()` 만들기 |
| 13 | Docker Qdrant에서도 A/B 점수(MRR@5 0.765)가 다시 나오는지 확인 |
| 14 | SQL 담당 전달용 적재 목록 저장 |
| 15 | 다음부터는 적재 없이 바로 검색하기 |

**실행 전 준비**

1. Qdrant 켜기 (PowerShell): `cd C:\dev\SKN36-3rd-3Team` (프로젝트 루트) → `docker compose up -d qdrant` → `http://localhost:6333/dashboard` 열리는지 확인
   - 2026-11-10부터 루트 `docker-compose.yml` + 로컬 `compose.override.yml`로 실행한다. `docker/qdrant/` 폴더에서 실행하면 컨테이너 이름이 충돌하니 쓰지 않는다.
2. 브랜치 확인: `git branch --show-current` → `feature/qdrant-embedding`
3. 패키지 (처음 한 번, 터미널): `uv pip install qdrant-client langchain-qdrant langchain-huggingface sentence-transformers numpy pandas`
   - KURE-v1은 처음 실행 때 약 2.2GB를 내려받는다.

## 1. 설정 + Qdrant 접속
- 경로·이름을 맨 위에 모아 두면 데이터 v2가 와도 **여기만 바꾸면** 된다.
- 컬렉션 이름 끝 `_v1` = 팀이 확정한 데이터 v1이라는 표시.

In [1]:
import json, uuid
import numpy as np
from qdrant_client import QdrantClient

DATA_ROOT = "../../data/rag/chunks/A_fixed"
CHUNK_FILES = [f"{DATA_ROOT}/policy_chunks.jsonl", f"{DATA_ROOT}/guideline_chunks.jsonl"]
VECTOR_PATH = "./results/cache/KURE-v1__A_fixed__4a35133a270bc941.npz"   # A/B 테스트에서 채점한 벡터
TEST_PATH = "./data/test_questions_v3.jsonl"
COLLECTION_NAME = "welfare_chunks_v1"
EXPECTED_COUNT = 3400
DIM = 1024

KURE_MODEL = "nlpai-lab/KURE-v1"
KURE_REV = "8b418a58414668e75532ed045c22d9ca018ae2b2"   # A/B 테스트와 같은 버전 (바꾸면 안 됨)

client = QdrantClient(host="localhost", port=6333)
print(client.get_collections())

collections=[CollectionDescription(name='policy_chunks_test')]


## 2. 청크 읽기 (metadata 펼치기)
- 청크 파일은 꼬리표가 `metadata` 안에 한 번 더 들어 있다 → **한 겹으로 펼쳐서** 담는다. A/B 테스트와 같은 구조라 지역 필터도 똑같이 걸린다.
- `policy_id`는 정책 청크의 바깥에만 있어서 metadata에 넣어 준다 → SQL `policies.policy_id`와 연결하는 키. (PDF 청크는 `section_id`·`document_id`가 이미 들어 있음)
- `assert`: 3,400개가 아니면 바로 멈추는 안전장치.

In [2]:
from langchain_core.documents import Document

documents = []
for path in CHUNK_FILES:
    with open(path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            meta = dict(row["metadata"])        # 지역·제목·parent_id 등 꼬리표
            meta["chunk_id"] = row["chunk_id"]
            meta["point_id"] = row["id"]
            if "policy_id" in row:               # 정책 청크: SQL policies와 연결하는 키
                meta["policy_id"] = row["policy_id"]
            documents.append(Document(page_content=row["text"], metadata=meta))

print("청크 수:", len(documents))
print("policy_id 있는 청크:", sum("policy_id" in d.metadata for d in documents))     # 662
print("section_id 있는 청크:", sum("section_id" in d.metadata for d in documents))    # 2,738
print(documents[0].page_content[:80])
print({k: documents[0].metadata[k] for k in ["chunk_id", "parent_id", "record_type", "sido_name", "sigungu_name"]})
assert len(documents) == EXPECTED_COUNT

청크 수: 3400
policy_id 있는 청크: 662
section_id 있는 청크: 2738
사업명: 서울형 장애인 개인예산제 시범사업
지역: 서울특별시
사업 개요: 장애인 당사자의 자기결정권 및 서비스 선택권 보장 강화
지원 대상: 서
{'chunk_id': 'policy::local:WLF00006795::fixed::0001', 'parent_id': 'local:WLF00006795', 'record_type': 'policy', 'sido_name': '서울특별시', 'sigungu_name': None}


## 3. 벡터 읽기 (A/B 테스트 캐시)
- A/B 테스트에서 실제로 채점한 **그 벡터**를 그대로 쓴다 (Neo4j 담당에게 준 패키지와도 같은 벡터) → GPU 없이 바로 적재.
- 모든 청크에 벡터가 있는지, 크기가 1(정규화)인지 확인한다.
- 캐시 파일 이름의 `4a35…`는 청크 파일 내용으로 만든 값이라, 청크 파일이 바뀌면 이 파일을 쓸 수 없다(이름이 달라짐).

In [3]:
z = np.load(VECTOR_PATH)
vectors = {cid: vec for cid, vec in zip(z["chunk_ids"].tolist(), z["vectors"])}
print("벡터 수:", len(vectors), "/ 길이:", z["vectors"].shape[1])

missing = [d.metadata["chunk_id"] for d in documents if d.metadata["chunk_id"] not in vectors]
norms = np.linalg.norm(z["vectors"], axis=1)
print("벡터 없는 청크:", len(missing))
print(f"벡터 크기: {norms.min():.4f} ~ {norms.max():.4f}")   # 1.0000이면 정규화됨
assert not missing and z["vectors"].shape == (EXPECTED_COUNT, DIM)

벡터 수: 3400 / 길이: 1024
벡터 없는 청크: 0
벡터 크기: 1.0000 ~ 1.0000


## 4. 컬렉션 만들기
- 연습(02) 때는 무조건 지우고 새로 만들었지만, 진짜 데이터는 실수로 지우지 않도록 **`RECREATE = True`일 때만** 지운다.
- 1024 · COSINE = A/B 테스트와 같은 설정.

In [4]:
from qdrant_client.models import VectorParams, Distance

RECREATE = False   # 처음부터 다시 만들고 싶을 때만 True

if client.collection_exists(COLLECTION_NAME):
    if RECREATE:
        client.delete_collection(COLLECTION_NAME)
        print("기존 컬렉션 삭제")
    else:
        print("이미 있음 → 지우지 않고 덮어쓰기(upsert)만 함")

if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=DIM, distance=Distance.COSINE),
    )
    print("컬렉션 생성:", COLLECTION_NAME)

컬렉션 생성: welfare_chunks_v1


## 5. 필터용 인덱스
- 책 뒤의 "찾아보기"처럼, 지역·종류로 거를 때 전부 뒤지지 않고 바로 찾게 해 준다.
- `parent_id`는 SQL · Neo4j와 연결할 때 쓰는 키(정책 = `policy_id`, PDF = `section_id`).

In [5]:
from qdrant_client.models import PayloadSchemaType

for field in ["sido_name", "sigungu_name", "record_type", "parent_id"]:
    client.create_payload_index(
        collection_name=COLLECTION_NAME,
        field_name=f"metadata.{field}",
        field_schema=PayloadSchemaType.KEYWORD,
    )
print("인덱스 완료")

인덱스 완료


## 6. 저장 (point 만들기)
- point = 번호 + 벡터 + payload(문장·꼬리표). 02와 같은 방식.
- 번호 = `chunk_id`로 만든 고정 UUID → 다시 실행해도 **덮어쓰기**(중복 없음). 청크 파일의 `id`와 같은지 `assert`로 확인.
- payload 이름 `page_content` / `metadata`는 LangChain `QdrantVectorStore`가 기본으로 읽는 이름.

In [6]:
from qdrant_client.models import PointStruct

points = [
    PointStruct(
        id=str(uuid.uuid5(uuid.NAMESPACE_URL, d.metadata["chunk_id"])),
        vector=vectors[d.metadata["chunk_id"]].tolist(),
        payload={"page_content": d.page_content, "metadata": d.metadata},
    )
    for d in documents
]
assert all(p.id == d.metadata["point_id"] for p, d in zip(points, documents))   # 청크 파일 id와 같은지

for i in range(0, len(points), 100):
    client.upsert(collection_name=COLLECTION_NAME, points=points[i:i + 100])
print("저장된 개수:", client.count(COLLECTION_NAME).count)

저장된 개수: 3400


## 7. 적재 확인
- 개수(3,400) · 설정(1024 · Cosine) 확인.
- 실제로 들어간 벡터가 캐시와 같은지 하나 꺼내서 비교 → 두 줄 숫자가 같아야 정상.

In [7]:
info = client.get_collection(COLLECTION_NAME)
print("상태:", info.status, "| 길이:", info.config.params.vectors.size, "| 거리:", info.config.params.vectors.distance)
assert client.count(COLLECTION_NAME).count == EXPECTED_COUNT

p = client.retrieve(COLLECTION_NAME, ids=[points[0].id], with_vectors=True)[0]
cid = p.payload["metadata"]["chunk_id"]
print(cid)
print("Qdrant 벡터 앞 3개:", [round(x, 4) for x in p.vector[:3]])
print("캐시 벡터 앞 3개  :", [round(float(x), 4) for x in vectors[cid][:3]])

상태: green | 길이: 1024 | 거리: Cosine
policy::local:WLF00006795::fixed::0001
Qdrant 벡터 앞 3개: [-0.0112, 0.0577, -0.0115]
캐시 벡터 앞 3개  : [-0.0112, 0.0577, -0.0115]


## 8. 질문 임베딩 모델 (KURE-v1)
- 질문도 **문서와 같은 모델 · 같은 버전 · 정규화**로 임베딩해야 같은 기준에서 비교된다 (하나라도 다르면 검색이 틀어짐).
- 설정 확인: 청크 하나를 지금 설정으로 다시 임베딩해서 저장된 벡터와 비교 → 코사인 유사도 **0.999 이상**이면 같은 설정. (GPU에서 만든 벡터를 CPU로 다시 계산하면 아주 작은 차이가 있어 1.0이 아닐 수 있음)

In [8]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings

KURE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = HuggingFaceEmbeddings(
    model_name=KURE_MODEL,
    model_kwargs={"device": KURE_DEVICE, "revision": KURE_REV},
    encode_kwargs={"normalize_embeddings": True},
)

d0 = documents[0]
v_now = np.array(embedding_model.embed_documents([d0.page_content])[0])
v_saved = np.array(vectors[d0.metadata["chunk_id"]])
cos = float(v_now @ v_saved)
print("장치:", KURE_DEVICE, "| 저장 벡터와 코사인 유사도:", round(cos, 6))
assert cos > 0.999, "모델·버전·정규화 설정이 A/B 테스트와 다름 → 확인"

C:\dev\SKN36-3rd-3Team\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   1%|          | 3/391 [00:00<00:13, 27.88it/s]

Loading weights:   2%|▏         | 6/391 [00:00<00:21, 18.30it/s]

Loading weights:   2%|▏         | 9/391 [00:00<00:23, 16.50it/s]

Loading weights:   3%|▎         | 11/391 [00:00<00:28, 13.22it/s]

... (7 outputs left out)

Loading weights:   8%|▊         | 31/391 [00:01<00:15, 23.41it/s]

Loading weights:   9%|▊         | 34/391 [00:02<00:14, 23.86it/s]

Loading weights: 100%|██████████| 391/391 [00:02<00:00, 191.69it/s]

장치: cpu | 저장 벡터와 코사인 유사도: 1.0


## 9. 지역 필터 (A/B 테스트와 같은 규칙)
- 규칙: **내 시도 또는 시도 없음(전국 PDF)** + **내 시군구 또는 시군구 없음(시도 전체 정책)**
- `IsEmptyCondition`은 값이 null · 키 없음 · 빈 리스트를 모두 "비어 있음"으로 잡는다 → PDF와 시도 전체 정책이 빠지지 않는다.
- 사는 곳을 모르면(`sido=None`) 필터 없이 검색한다.
- 검수 필터(`approved`)는 아직 넣지 않는다: 지금 데이터는 `approved`가 0건이라 걸면 결과가 0개가 된다.

In [9]:
from qdrant_client.models import Filter, FieldCondition, MatchValue, IsEmptyCondition, PayloadField

def match_or_empty(field, value):
    key = f"metadata.{field}"
    return Filter(should=[
        FieldCondition(key=key, match=MatchValue(value=value)),
        IsEmptyCondition(is_empty=PayloadField(key=key)),
    ])

def region_filter(sido=None, sigungu=None):
    if not sido:
        return None
    conditions = [match_or_empty("sido_name", sido)]
    if sigungu:
        conditions.append(match_or_empty("sigungu_name", sigungu))
    return Filter(must=conditions)

print(region_filter("서울특별시", "마포구"))

should=None min_should=None must=[Filter(should=[FieldCondition(key='metadata.sido_name', match=MatchValue(value='서울특별시'), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None), IsEmptyCondition(is_empty=PayloadField(key='metadata.sido_name'))], min_should=None, must=None, must_not=None), Filter(should=[FieldCondition(key='metadata.sigungu_name', match=MatchValue(value='마포구'), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None), IsEmptyCondition(is_empty=PayloadField(key='metadata.sigungu_name'))], min_should=None, must=None, must_not=None)] must_not=None


## 10. vectorstore 연결
- 이미 적재한 컬렉션에 LangChain을 연결한다 (강사님 코드 방식). 질문은 8번의 KURE 모델로 임베딩된다.

In [10]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embedding_model,
)
print("vectorstore 연결 완료")

vectorstore 연결 완료


## 11. 검색 함수 `search_policies()` (팀 전달용)
- 질문 + 사는 곳 → 근거 청크 목록. **연결 담당 · Neo4j 담당 · 나중의 에이전트가 이 함수 하나만 부르면 된다.**
- 출력 모양은 팀이 합의한 공통 형식: `parent_id`, `chunk_id`, `score`, `text`, `source="qdrant"` (+ 출처 정보)
- `parent_id`로 SQL(정책·PDF 상세)과 Neo4j(관계 확장)에 연결한다.
- `score`는 코사인 유사도(1에 가까울수록 비슷).

In [11]:
def search_policies(question, sido=None, sigungu=None, k=5):
    results = vectorstore.similarity_search_with_score(question, k=k, filter=region_filter(sido, sigungu))
    out = []
    for doc, score in results:
        m = doc.metadata
        out.append({
            "parent_id": m.get("parent_id"),          # 정책 = policy_id, PDF = section_id
            "chunk_id": m.get("chunk_id"),
            "record_type": m.get("record_type"),      # policy / guideline
            "title": m.get("title"),
            "score": round(float(score), 3),
            "text": doc.page_content,
            "sido_name": m.get("sido_name"),
            "sigungu_name": m.get("sigungu_name"),
            "source_url": m.get("source_url"),
            "page_start": m.get("page_start"),        # PDF만
            "page_end": m.get("page_end"),
            "source": "qdrant",
        })
    return out

## 12. 사용 예시
- 지역 질문은 내 지역 정책 + 시도 전체 정책 + 전국 PDF만 나와야 한다.

In [12]:
examples = [
    ("마포구 사는데 다리가 약해서 밀고 다니는 거 하나 있으면 좋겠어", "서울특별시", "마포구"),
    ("여주시에 사는데 어르신 버스비 지원받으려면 어디로 가야 돼요?", "경기도", "여주시"),
    ("부부가 둘 다 기초연금 받으면 깎인다던데 진짜야?", None, None),
]
for q, sido, sgg in examples:
    print(f"\n질문: {q}  (사는 곳: {sido} {sgg or ''})")
    for r in search_policies(q, sido, sgg, k=3):
        where = f"{r['sido_name'] or '전국'} {r['sigungu_name'] or ''}".strip()
        print(f"  {r['score']} | {r['record_type']:9} | {where:12} | {r['title']} | {r['parent_id']}")


질문: 마포구 사는데 다리가 약해서 밀고 다니는 거 하나 있으면 좋겠어  (사는 곳: 서울특별시 마포구)


  0.548 | policy    | 서울특별시 마포구    | 저소득 노인 보행보조차 지원 | local:WLF00000696
  0.522 | policy    | 서울특별시 마포구    | 마포 징검다리 주택사업 | local:WLF00001859
  0.492 | guideline | 전국           | 2026 나에게 힘이 되는 복지서비스 | welfare_services_2026_p0046_01

질문: 여주시에 사는데 어르신 버스비 지원받으려면 어디로 가야 돼요?  (사는 곳: 경기도 여주시)


  0.782 | policy    | 경기도 여주시      | 여주시 어르신 시내버스 무상교통 지원사업 | local:WLF00005991
  0.714 | policy    | 경기도 여주시      | 노인 성인용보행기 비용 지원 | local:WLF00003819
  0.647 | policy    | 경기도 여주시      | 여주시 부양자 부양지원금 지원 사업 | local:WLF00004774

질문: 부부가 둘 다 기초연금 받으면 깎인다던데 진짜야?  (사는 곳: None )


  0.689 | guideline | 전국           | 2026년 기초연금 사업안내 | basic_pension_2026_p0138_01
  0.667 | guideline | 전국           | 2026년 기초연금 사업안내 | basic_pension_2026_p0131_01
  0.663 | guideline | 전국           | 2026년 기초연금 사업안내 | basic_pension_2026_p0024_01


## 13. A/B 결과 재현 확인 (Docker Qdrant)
- A/B 테스트는 메모리 Qdrant(`:memory:`)에서 했다. 실제 서비스용 Docker Qdrant에서도 같은 점수가 나오는지 확인한다.
- 채점 방식은 A/B 테스트와 같다: 상위 5개 청크 중 **정답 부모(`parent_id`)가 처음 나온 순위**로 Hit@5 · MRR@5.
- 기대: MRR@5 ≈ **0.765**, Hit@5 ≈ **0.885** (113개).
- 아주 작은 차이(±0.01)는 정상일 수 있다: Docker Qdrant는 빠른 근사 검색(HNSW)을 쓰고, 질문 임베딩을 GPU가 아닌 CPU로 계산했을 수 있기 때문. 차이가 크면 설정이 다른 것이니 멈추고 확인한다.

In [13]:
import pandas as pd

with open(TEST_PATH, encoding="utf-8") as f:
    questions = [json.loads(line) for line in f if line.strip()]
scoring = [q for q in questions if not q["q_type"].startswith("함정")]
print("채점 질문:", len(scoring))   # 113

rows = []
for q in scoring:
    results = search_policies(q["question"], q.get("user_sido"), q.get("user_sigungu"), k=5)
    expected = set(q["expected_ids"])
    rank = next((i for i, r in enumerate(results, start=1) if r["parent_id"] in expected), None)
    rows.append({"qid": q["qid"], "q_type": q["q_type"], "rank": rank})

df = pd.DataFrame(rows)
hit5 = df["rank"].notna().mean()
mrr5 = (1 / df["rank"]).fillna(0).mean()
print(f"Hit@5 = {hit5:.3f} (A/B 0.885) | MRR@5 = {mrr5:.3f} (A/B 0.765)")
print(df.assign(rr=(1 / df["rank"]).fillna(0)).groupby("q_type")["rr"].mean().round(3))

채점 질문: 113


Hit@5 = 0.885 (A/B 0.885) | MRR@5 = 0.765 (A/B 0.765)
q_type
PDF근거    0.639
시도전체     0.583
지역정책     0.988
지역짝      0.708
Name: rr, dtype: float64


## 14. SQL 담당 전달용 적재 목록
- SQL `policy_chunks` / `guideline_chunks` 테이블의 `embedding_status`를 갱신할 수 있게, Qdrant에 들어간 청크 목록을 파일로 남긴다.
- `chunk_id`로 SQL 행과 연결, `point_id`는 Qdrant 번호.

In [14]:
loaded = pd.DataFrame([{
    "chunk_id": d.metadata["chunk_id"],
    "point_id": d.metadata["point_id"],
    "parent_id": d.metadata["parent_id"],
    "record_type": d.metadata["record_type"],
    "collection": COLLECTION_NAME,
    "embedding_model": f"{KURE_MODEL}@{KURE_REV[:8]}",
} for d in documents])
loaded.to_csv("./results/qdrant_v1_loaded_chunks.csv", index=False, encoding="utf-8-sig")
print(len(loaded), "건 저장 → results/qdrant_v1_loaded_chunks.csv")

3400 건 저장 → results/qdrant_v1_loaded_chunks.csv


## 15. 다음부터는: 적재 없이 바로 검색
- Qdrant 데이터는 Docker 볼륨에 남아 있어서 노트북을 꺼도 사라지지 않는다.
- 다음부터는 **셀 1 → 8 → 9 → 10 → 11**만 실행하면 바로 `search_policies()`를 쓸 수 있다 (적재 2~7은 데이터가 바뀔 때만).

In [15]:
print("컬렉션:", COLLECTION_NAME, "| 저장된 개수:", client.count(COLLECTION_NAME).count)
search_policies("혼자 사는 어르신 안부 확인 서비스", "전라남도", "목포시", k=3)

컬렉션: welfare_chunks_v1 | 저장된 개수: 3400


[{'parent_id': 'welfare_services_2026_p0210_01',
  'chunk_id': 'guideline::welfare_services_2026_p0210_01::fixed::0001',
  'record_type': 'guideline',
  'title': '2026 나에게 힘이 되는 복지서비스',
  'score': 0.657,
  'text': '목차: 07 노령층 지원 > 어르신을 돌봐 드릴 일손이 필요할 때 > 독거노인･장애인 응급안전 안심서비스\n\n07 노령층 지원\n210 2026 나에게 힘이 되는 복지서비스\n독거노인･장애인\xa0 응급안전\xa0 안심서비스\nⅠ 대상 상시보호가 필요한 어르신 및 장애인\n∙ (독거가구) 주민등록상 거주지와 동거자 유무, 소득과 관계없이 실제로 혼자 살고 있는 65세 이상의 어르신\n∙ (노인 2인가구) 65세 이상 2인으로 구성되며, 기초생활수급자, 차상위, 기초연금수급자 중 한 명이 질환을 \n앓고 있거나 거동이 불편한 경우나 모두 75세 이상인 경우\n∙ (조손가구) 노인 1인의 경우는 독거가구 기준과 동일하며, 2인의 경우는 노인 2인가구와 동일하게 \n가능\n∙ (장애인) 장애인활동지원 수급자이면서 독거 또는 취약가구, 수급자 외 장애인의 경우 기초지자체장이 \n독거 또는 취약가구 등의 생활 여건을 고려해 상시 보호가 필요하다고 인정하는 경우 가능\n※ 단, 24시간 활동지원서비스를 받고 있는 장애인은 선정 제외\nⅠ 내용 어르신과 장애인 가정에 화재센서, 활동감지기 및 응급호출기를 설치하여 응급상황 발생 시 신속하게 대처할 \n수 있도록 119에 신고\n응급호출기\n응급버튼을 누르면\n소방서(119)로 연결돼요\n소방서로 연결\n저장된 비상연락처\n(자녀 등)로 전화\n게이트웨이\n활동량감지기\n움직임이 없으면\n알려줘요\n화재감지기\n불이 나면 연기를 감지해서\n소방서(119)로 연결돼요\n출입감지기\n문이 열리고 닫힌 걸\n알려줘요\nⅠ 방법 사업대상자 추천 및 대상